<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/14-Adding_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conversational RAG with Chat Engine and ReAct Agent

This notebook demonstrates how to build a multi-turn conversational RAG system using LlamaIndex chat engines and a ReAct agent.

## What You Will Learn
- How to set up a LlamaIndex chat engine with memory for multi-turn conversations
- How memory persists across turns and how to reset it
- How to stream tokens as they are generated
- How the `condense_question` mode rewrites follow-up questions using chat history
- How to build a ReAct agent that uses a query engine as a reasoning tool

## Prerequisites
- Basic Python and Jupyter notebook familiarity
- Understanding of vector stores and RAG pipelines (covered in earlier notebooks)
- An OpenAI API key with access to GPT-5 models

## Install Packages and Setup Variables


In [ ]:
!pip install -qU llama-index==0.14.19 openai==2.8.1 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 \
                 opentelemetry-api==1.38.0 llama-index-embeddings-openai==0.6.0 jedi==0.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Initialize Models


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [ ]:
# Download the vector store from Hugging Face Hub.
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [ ]:
!unzip -o vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Load the vector store from local storage.
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

# Helper: Display Response and Sources


In [ ]:
# A simple function to display the response and its sources.
def display_res(response):
    print("Response:\n\t", response.response.replace("\n", ""))

    print("Sources:")
    if response.source_nodes:
        for src in response.source_nodes:
            print("\tNode ID\t", src.node_id)
            print("\tText\t", src.text)
            print("\tScore\t", src.score)
            print("\t" + "-_" * 20)
    else:
        print("\tNo sources used!")

# Chat Engine


In [ ]:
# Define the chat engine using the vector index.
chat_engine = vector_index.as_chat_engine()

In [ ]:
# First question:
response = chat_engine.chat("Use the tool to answer, how does parameter efficient finetuning work?")

display_res(response)

Response:
	 Short answerParameter-efficient fine-tuning (PEFT) adapts a large pretrained model to a new task by changing only a small, compact set of parameters (or an auxiliary set) instead of updating the whole model. The goal is to: (1) reduce the number of trainable parameters, (2) lower storage/compute for many task-specific adapters, and (3) often retain most of the base model’s capabilities.How it works — common patterns- Keep base model weights frozen. Only add or learn a small, separate set of parameters so the original model remains unchanged.- Inject lightweight modules or parameterizations into particular layers (often linear layers) so the model behavior can change without full-weight updates.- At inference you either: (a) apply the learned deltas/adapter modules to the frozen model, or (b) merge them into the base weights if you need a single consolidated model.Representative techniques (from the provided docs)- LoRA (low-rank adaptation)  - Idea: represent the weight upd

In [ ]:
# Second question:
response = chat_engine.chat("Could you tell me a joke?")
display_res(response)

Response:
	 Sure — here's one for you:Why couldn't the bicycle stand up by itself?Because it was two tired!Want another (kangaroo-themed or something else)?
Sources:
	Node ID	 c4d6c614-e8ba-4aa4-92df-a2a2d4456717
	Text	 the possum it could be done.\n\n- It was on its way to a poultry farmers\' convention.\n\nThe joke plays on the double meaning of "the other side" - literally crossing the road to the other side, or the "other side" meaning the afterlife. So it\'s an anti-joke, with a silly or unexpected pun as the answer.' additional_kwargs={} example=FalseWe can use our "LLM with Fallbacks" as we would a normal LLM.```pythonfrom langchain_core.prompts import ChatPromptTemplateprompt = ChatPromptTemplate.from_messages(    [        (            "system",            "You're a nice assistant who always includes a compliment in your response",        ),        ("human", "Why did the {animal} cross the road"),    ])chain = prompt | llmwith patch("openai.resources.chat.completions.Completion

In [ ]:
# Third question: check if the engine recalls previous interactions.
response = chat_engine.chat("What was the first question I asked?")
display_res(response)

Response:
	 Your first question was: "Use the tool to answer, how does parameter efficient finetuning work?"
Sources:
	Node ID	 91ee09f9-e626-4e26-93e9-1dc846ec265f
	Text	 import WebBaseLoaderfrom langchain_community.vectorstores import FAISSfrom langchain_openai.chat_models import ChatOpenAIfrom langchain_openai.embeddings import OpenAIEmbeddingsfrom langchain_text_splitters import RecursiveCharacterTextSplitterloader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")data = loader.load()# Splittext_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)all_splits = text_splitter.split_documents(data)# Store splitsvectorstore = FAISS.from_documents(documents=all_splits, embedding=OpenAIEmbeddings())# LLMllm = ChatOpenAI()```## Legacy<details open>```pythonfrom langchain.chains import ConversationalRetrievalChainfrom langchain_core.prompts import ChatPromptTemplatecondense_question_template = """Given the following conversation and a follow up que

In [ ]:
# Reset the session to clear the memory.
chat_engine.reset()

In [ ]:
# Fourth question: the engine should not recall previous interactions after reset.
response = chat_engine.chat("What was the first question I asked?")
display_res(response)

Response:
	 don't know
Sources:
	Node ID	 b71bd703-51b9-4ff9-9694-b0823ad1f178
	Text	 Node ID: adb6b7ce-49bb-4961-8506-37082c02a389    Text: What I Worked On  February 2021  Before college the two main    things I worked on, outside of school, were writing and programming. I    didn't write essays. I wrote what beginning writers were supposed to    write then, and probably still are: short stories. My stories were    awful. They had hardly any plot, just characters with strong feelings,    which I ...    Score:  0.802        Node ID: e39be1fe-32d0-456e-b211-4efabd191108    Text: Except for a few officially anointed thinkers who went to the    right parties in New York, the only people allowed to publish essays    were specialists writing about their specialties. There were so many    essays that had never been written, because there had been no way to    publish them. Now they could be, and I was going to write them. [12]    I've wor...    Score:  0.799    ```pythonresponse = query_eng

## Optional: Inspecting Chat Memory

In [ ]:
# LlamaIndex chat engines store conversation history in a memory buffer.
# This lets you verify what context is available to the model at each turn.
try:
    # Re-initialize a fresh chat engine to demonstrate memory tracking
    demo_engine = vector_index.as_chat_engine(verbose=False)

    turns = [
        "What is LLaMA 2?",
        "How many parameters does it have?",
        "Can you summarize what we discussed?",
    ]

    print("=== Conversation with Memory Inspection ===\n")
    for i, msg in enumerate(turns, 1):
        response = demo_engine.chat(msg)
        history = demo_engine.chat_history
        print(f"Turn {i}: {msg}")
        print(f"  Response : {str(response)[:200]}...")
        print(f"  Memory   : {len(history)} message(s) in history")
        print()
except Exception as e:
    print(f"Could not inspect chat memory: {e}")

=== Conversation with Memory Inspection ===

Turn 1: What is LLaMA 2?
  Response : Llama2 (LLaMA 2) is a follow-up release from Meta AI built on the original LLaMA model. According to the provided documentation, Llama2 includes architectural tweaks—most notably Grouped Query Attenti...
  Memory   : 2 message(s) in history

Turn 2: How many parameters does it have?
  Response : I don't know — the provided documents don't state how many parameters LLaMA 2 has. (The original LLaMA family is described as ranging from 7B to 65B parameters.)...
  Memory   : 4 message(s) in history

Turn 3: Can you summarize what we discussed?
  Response : Summary of our conversation and the relevant doc details:

- What is LLaMA 2?
  - LLaMA 2 (Llama2) is a follow-up release from Meta AI built on the original LLaMA. It introduces architectural tweaks (...
  Memory   : 6 message(s) in history



# Streaming


In [ ]:
# Stream tokens as soon as they are available instead of waiting for the model to finish generation.
streaming_response = chat_engine.stream_chat(
    "Write a paragraph explaining how RAG and PEFT work, and highlight the differences between them."
)
streaming_response.print_response_stream()

don't know

## Condense Question


Enhance the input prompt by looking at the previous chat history along with the present question. The refined prompt can then be used to fetch the relevant nodes.


In [ ]:
# Define the GPT-5 model that will be used by the chat engine to improve the query.
gpt5 = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})

In [ ]:
chat_engine = vector_index.as_chat_engine(
    chat_mode="condense_question", llm=gpt5, verbose=True
)

In [ ]:
response = chat_engine.chat(
    "How does Retrieval-Augmented Generation (RAG) work, and which problem does it solve?"
)
display_res(response)

Querying with: How does Retrieval-Augmented Generation (RAG) work, and which problem does it solve?
Response:
	 Retrieval-Augmented Generation (RAG) is a hybrid approach that combines an information retrieval system with a generative large language model (LLM) to produce more accurate, up-to-date, and context-grounded responses.How it works (high-level flow)- Query classification: decide whether the input requires retrieval (i.e., needs external documents) or can be handled by the model alone.- Retrieval / Indexing & Searching: index external knowledge sources (sparse inverted indexes or dense vector encodings) and search them for query-relevant documents or chunks.- Reranking: refine the ordering of retrieved items to surface the most relevant passages.- Repacking: organize or concatenate the selected passages into a structured context to feed the generator.- Summarization / Generation: the LLM uses the repacked context plus the query to generate the final answer; prompting techniques

## Optional: What Question Did the Condense Mode Generate?

In [ ]:
# The "condense_question" mode rewrites the user's question using chat history
# before sending it to the retriever. This helps retrieve more relevant context
# for follow-up questions that use pronouns or refer to earlier topics.
print("=== Condense Question Mode Explained ===\n")
print("Mode: condense_question")
print("What it does:")
print("  1. Takes the current question + full chat history")
print("  2. Sends both to the LLM (gpt5) to generate a standalone question")
print("  3. Uses that condensed question to retrieve context from the vector store")
print("  4. Generates the final answer from the retrieved context\n")
print("Example:")
print("  History: 'How does RAG work?'")
print("  Follow-up: 'How does that compare to fine-tuning?'")
print("  Condensed: 'How does Retrieval-Augmented Generation compare to fine-tuning?'")
print("\nThis ensures the retriever gets a complete, unambiguous query every time.")

=== Condense Question Mode Explained ===

Mode: condense_question
What it does:
  1. Takes the current question + full chat history
  2. Sends both to the LLM (gpt5) to generate a standalone question
  3. Uses that condensed question to retrieve context from the vector store
  4. Generates the final answer from the retrieved context

Example:
  History: 'How does RAG work?'
  Follow-up: 'How does that compare to fine-tuning?'
  Condensed: 'How does Retrieval-Augmented Generation compare to fine-tuning?'

This ensures the retriever gets a complete, unambiguous query every time.


## ReAct


In [ ]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.tools import QueryEngineTool

query_engine = vector_index.as_query_engine()

tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="ReAct Agent",
    description="Answer questions using the vector index; pass plain text queries.",
)

agent = ReActAgent(
    tools=[tool],
    verbose=True

)

# Context to hold this session's state.
ctx = Context(agent)

handler = agent.run("Which company developed Claude 3.5 Sonnet, and what is its primary application?", ctx=ctx, max_iterations=4)

In [ ]:
response = await handler
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='Which company developed Claude 3.5 Sonnet, and wha...', chat_history=None, memory=None, max_iterations=4, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Which company developed Claude 3.5 Sonnet, and what is its primary ap...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Which company developed Claude 3.5 Sonnet, and what is its primary ap...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_typ